In [ ]:
# !gdown 13k17SjgQHZCO-1Ctr3DY_bW6DGvQZZie
# !unzip -q /kaggle/working/model_outputs.zip
# !rm /kaggle/working/model_outputs.zip
!pip install segmentation_models_pytorch -q
!pip install clearml

In [ ]:
import torch
from torch import nn
import segmentation_models_pytorch as smp


class QueryAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
        )
        self.norm3 = nn.LayerNorm(embed_dim)

    def forward(self, queries, tokens, return_attn=False):
        attn_out, attn_weights = self.cross_attn(
            queries, tokens, tokens, need_weights=return_attn, average_attn_weights=True
        )
        queries = self.norm1(queries + attn_out)

        self_out, _ = self.self_attn(queries, queries, queries)
        queries = self.norm2(queries + self_out)

        queries = self.norm3(queries + self.ffn(queries))
        return (queries, attn_weights) if return_attn else queries


class OcclusionClassifier(nn.Module):
    def __init__(self, encoder: nn.Module, num_classes=8, embed_dim=128,
                 num_heads=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.encoder = encoder
        in_channels = encoder.out_channels[-1]
        self.input_proj = nn.Conv2d(in_channels, embed_dim, kernel_size=1)
        self.class_queries = nn.Parameter(torch.randn(num_classes, embed_dim) * 0.02)
        self.layers = nn.ModuleList([
            QueryAttentionBlock(embed_dim, num_heads, dropout) for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, x, return_attn=False):
        feat = self.encoder(x)[-1]
        feat = self.input_proj(feat)
        B, D, H, W = feat.shape
        tokens = feat.flatten(2).permute(0, 2, 1)
        queries = self.class_queries.unsqueeze(0).expand(B, -1, -1)

        last_attn = None
        for layer in self.layers:
            if return_attn:
                queries, last_attn = layer(queries, tokens, return_attn=True)
            else:
                queries = layer(queries, tokens)

        logits = self.classifier(queries).squeeze(-1)

        if return_attn:
            # last_attn: [B, num_classes, H*W] -> [B, num_classes, H, W]
            attn_map = last_attn.reshape(B, -1, H, W)
            return logits, attn_map
        return logits

In [ ]:
ROOT = "/kaggle/input/datasets/saveliymazovatov/fpn18"
 
CHECKPOINTS = {
    "fpn_resnet18_torch_cross_entropy_all_files": {
        "path": f"{ROOT}/fpn_resnet18_torch_cross_entropy_all_files/model/fpn_resnet18/fpn_resnet18.ckpt",
        "arch": "FPN",
        "encoder_name": "resnet18",
    },
    # "linknet_resnet18_torch_cross_entropy_all_files": {
    #     "path": f"{ROOT}/linknet_resnet18_torch_cross_entropy_all_files/model/linknet_resnet18/linknet_resnet18.ckpt",
    #     "arch": "Linknet",
    #     "encoder_name": "resnet18",
    # },
    # "manet_resnet18_torch_cross_entropy_all_files": {
    #     "path": f"{ROOT}/manet_resnet18_torch_cross_entropy_all_files/model/manet_resnet18/manet_resnet18.ckpt",
    #     "arch": "MAnet",
    #     "encoder_name": "resnet18",
    # },
    # "pspnet_resnet18_torch_cross_entropy_all_files": {
    #     "path": f"{ROOT}/pspnet_resnet18_torch_cross_entropy_all_files/model/pspnet_resnet18/pspnet_resnet18.ckpt",
    #     "arch": "PSPNet",
    #     "encoder_name": "resnet18",
    # },
    # "unet_resnet18_torch_cross_entropy_all_files": {
    #     "path": f"{ROOT}/unet_resnet18_torch_cross_entropy_all_files/model/unet_resnet18/unet_resnet18.ckpt",
    #     "arch": "Unet",
    #     "encoder_name": "resnet18",
    # },
}

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def stats_path(model_key, filename):
    return Path(ROOT) / model_key / "evaluations" / "stats" / filename

general_long = []
for key, cfg in CHECKPOINTS.items():
    df = pd.read_csv(stats_path(key, "general_stats.csv"))

    if {"metric", "value"}.issubset(df.columns):
        long_df = df[["metric", "value"]].copy()
    else:
        long_df = df.iloc[0:1].melt(var_name="metric", value_name="value")

    long_df["model"] = key
    long_df["arch"] = cfg["arch"]
    long_df["encoder"] = cfg["encoder_name"]
    general_long.append(long_df)

general_df = pd.concat(general_long, ignore_index=True)
general_wide = general_df.pivot(index="arch", columns="metric", values="value")
print(general_wide)
# Возьмем энкодер FPN , тк он лучше всего показывает себя в сегментации

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp


def build_encoder_classifier(
    arch: str,
    encoder_name: str,
    ckpt_path: str,
    num_classes: int = 7,
    embed_dim: int = 128,
    num_heads: int = 4,
    num_layers: int = 2,
    dropout: float = 0.2,
    freeze_encoder: bool = False,
    device: str | torch.device = "cpu",
) -> OcclusionClassifier:
    seg_model = getattr(smp, arch)(encoder_name=encoder_name, encoder_weights=None)

    raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)["state_dict"]
    state_dict = {k[len("model."):]: v for k, v in raw.items() if k.startswith("model.")}

    missing, unexpected = seg_model.load_state_dict(state_dict, strict=False)
    encoder_missing = [k for k in missing if k.startswith("encoder.")]
    print(f"[{arch}/{encoder_name}] encoder missing={len(encoder_missing)}, "
          f"decoder/head missing={len(missing) - len(encoder_missing)}, unexpected={len(unexpected)}")

    encoder = seg_model.encoder
    if freeze_encoder:
        for p in encoder.parameters():
            p.requires_grad = False

    model = OcclusionClassifier(
        encoder, num_classes=num_classes, embed_dim=embed_dim,
        num_heads=num_heads, num_layers=num_layers, dropout=dropout,
    )
    return model.to(device)


models = {
    key: build_encoder_classifier(cfg["arch"], cfg["encoder_name"], cfg["path"], freeze_encoder=True, device="cpu")
    for key, cfg in CHECKPOINTS.items()
}

In [ ]:
import torchvision.transforms as T

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),          # горизонтально ок — дорога симметрична слева-направо
    T.RandomRotation(degrees=20),            # без vertical flip, поворот в пределах ±20°
    T.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2,
        hue=0.05,
    ),
    T.RandomAutocontrast(p=0.2),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
import os
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T


DATA_ROOT = Path("/kaggle/input/datasets/saveliymazovatov/occlusions/Datasets 2")
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

DATASET_STRUCTURE = {
    "Clean": {
        "train": ["/kaggle/input/datasets/saveliymazovatov/clean-dataset/clean/train"],
        "test":  ["/kaggle/input/datasets/saveliymazovatov/clean-dataset/clean/test"],
    },
    "DaytimeFlare": {
        "train": ["DaytimeFlare/train"],
        "test":  ["DaytimeFlare/test"],
    },
    "Fog": {
        "train": ["Fog/train"],
        "test":  ["Fog/test"],
    },
    "MotionBlur": {
        "train": ["MotionBlur/train"],
        "test":  ["MotionBlur/test"],
    },
    "NighttimeFlare": {
        "train": ["NighttimeFlare/train"],
        "test":  ["NighttimeFlare/test"],
    },
    "Raindrops": {
        "train": ["Raindrops/train"],
        "test":  ["Raindrops/test"],
    },
    # "Reflections": {
    #     "train": ["Reflections/train"],
    #     "test":  ["Reflections/test"],
    # },
    # "Snowflakes": {
    #     "train": ["Snowflakes/train"],
    #     "test":  ["Snowflakes/test"],
    # },
    "Soil": {
        "train": ["Soil/train"],
        "test":  ["Soil/test"],
    },
}

CLASSES = [c for c in DATASET_STRUCTURE.keys() if c != "Clean"]
print(f"Классы ({len(CLASSES)}): {CLASSES}")


def list_images(rel_dir: str) -> list[Path]:
    full_dir = DATA_ROOT / rel_dir
    if not full_dir.exists():
        print(f"  ⚠ нет папки: {full_dir}")
        return []
    return [full_dir / f for f in os.listdir(full_dir) if Path(f).suffix.lower() in IMG_EXTS]


def build_index():
    train_samples, test_samples = [], []
    for class_name, splits in DATASET_STRUCTURE.items():
        n_train = n_test = 0
        for rel_dir in splits["train"]:
            files = list_images(rel_dir)
            train_samples += [(p, class_name) for p in files]
            n_train += len(files)
        for rel_dir in splits["test"]:
            files = list_images(rel_dir)
            test_samples += [(p, class_name) for p in files]
            n_test += len(files)
        print(f"  [{class_name}] train={n_train}, test={n_test}")
    return train_samples, test_samples


train_samples, test_samples = build_index()
print(f"\nTrain: {len(train_samples)}, Test: {len(test_samples)}")
print("Train по классам:", Counter(c for _, c in train_samples))

In [ ]:
# Костыль тк датасет несбалансированный
import random

MAX_PER_CLASS = 800

def cap_per_class(samples, max_per_class, seed=42):
    rng = random.Random(seed)
    by_class = {}
    for p, c in samples:
        by_class.setdefault(c, []).append(p)

    capped = []
    for c, paths in by_class.items():
        if len(paths) > max_per_class:
            paths = rng.sample(paths, max_per_class)
        capped += [(p, c) for p in paths]
    return capped


train_samples_full = train_samples          # держим оригинал на случай если понадобится откатить
train_samples = cap_per_class(train_samples_full, MAX_PER_CLASS)

print("До ограничения:", Counter(c for _, c in train_samples_full))
print("После ограничения:", Counter(c for _, c in train_samples))

In [ ]:
class OcclusionDataset(Dataset):
    def __init__(self, samples, classes, transform):
        self.samples = samples
        self.classes = classes
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, class_name = self.samples[idx]
        image = self.transform(Image.open(path).convert("RGB"))

        target = torch.zeros(len(self.classes))
        mask = torch.zeros(len(self.classes))

        if class_name == "Clean":
            mask[:] = 1.0                      # подтверждённо: окклюзий нет вообще
        else:
            i = self.class_to_idx[class_name]
            target[i] = 1.0
            mask[i] = 1.0                      # известна только эта позиция, остальные — PU

        return image, target, mask

train_dataset = OcclusionDataset(train_samples, CLASSES, train_transform)
test_dataset = OcclusionDataset(test_samples, CLASSES, test_transform) if test_samples else None

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True) if test_dataset else None

In [ ]:
from torch.optim.lr_scheduler import LambdaLR
import torch.nn.functional as F
import math

NUM_EPOCHS = 30
device = "cuda" if torch.cuda.is_available() else "cpu"

model = build_encoder_classifier(
    arch="FPN",
    encoder_name="resnet18",
    ckpt_path=CHECKPOINTS["fpn_resnet18_torch_cross_entropy_all_files"]["path"],
    num_classes=len(CLASSES),
    freeze_encoder=False,
    device=device,
)

def focal_bce_loss(logits, targets, gamma=2.0, alpha=None, reduction="mean"):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = torch.exp(-bce)                      # эквивалент p, если target=1, и (1-p), если target=0
    loss = (1 - p_t) ** gamma * bce

    if alpha is not None:
        # alpha — вес позитивного класса (per-class тензор [C] или скаляр)
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        loss = alpha_t * loss

    return loss.mean() if reduction == "mean" else loss.sum() if reduction == "sum" else loss


def focal_loss(logits, targets):
    return focal_bce_loss(logits, targets, gamma=2.0)


encoder_params = list(model.encoder.parameters())
head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]

optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": 1e-5},
    {"params": head_params,    "lr": 1e-4},
])
# ↑ убрана строка "optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)" — она затирала группы

steps_per_epoch = len(train_loader)
total_steps = NUM_EPOCHS * steps_per_epoch
warmup_steps = steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)

In [ ]:
# Ключи берём из секретов среды (Kaggle: Add-ons -> Secrets, Colab: панель ключей).
# В ноутбук их не вписываем.
%env CLEARML_WEB_HOST=https://app.occlusionnet.duckdns.org
%env CLEARML_API_HOST=https://api.occlusionnet.duckdns.org
%env CLEARML_FILES_HOST=https://files.occlusionnet.duckdns.org
%env CLEARML_API_ACCESS_KEY=
%env CLEARML_API_SECRET_KEY=


In [ ]:
import numpy as np
import torch
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from clearml import Task
from tqdm.notebook import tqdm

task = Task.init(project_name="OcclusionNet", task_name="FPN_resnet18_multilabel_Saveliy")
logger = task.get_logger()
task.connect({"num_epochs": NUM_EPOCHS, "lr": optimizer.param_groups[0]["lr"], "classes": CLASSES})


def compute_grad_norm(model):
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return total ** 0.5


@torch.no_grad()
def evaluate(model, loader, classes, device, loss_fn, threshold=0.5):
    model.eval()
    all_logits, all_targets = [], []
    running_loss = 0.0

    for images, targets, mask in tqdm(loader, desc="  val", leave=False):
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        running_loss += loss_fn(logits, targets).item() * images.size(0)   # без mask
        all_logits.append(logits.cpu())
        all_targets.append(targets.cpu())

    val_loss = running_loss / len(loader.dataset)

    logits = torch.cat(all_logits)
    targets = torch.cat(all_targets)
    preds = (torch.sigmoid(logits) > threshold).float()

    per_class = {}
    precisions, recalls, f1s, accs = [], [], [], []

    for i, cls_name in enumerate(classes):
        y_true = targets[:, i].numpy()   # было targets[valid, i] — теперь берём всё
        y_pred = preds[:, i].numpy()     # было preds[valid, i]

        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        acc = accuracy_score(y_true, y_pred)

        per_class[cls_name] = {"precision": p, "recall": r, "f1": f1, "accuracy": acc}
        precisions.append(p); recalls.append(r); f1s.append(f1); accs.append(acc)

    macro = {
        "precision": float(np.mean(precisions)) if precisions else 0.0,
        "recall": float(np.mean(recalls)) if recalls else 0.0,
        "f1": float(np.mean(f1s)) if f1s else 0.0,
        "accuracy": float(np.mean(accs)) if accs else 0.0,
    }
    return val_loss, macro, per_class

In [ ]:
import torch.nn.functional as F
from matplotlib.gridspec import GridSpec

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

INFERENCE_DIR = Path("/kaggle/input/datasets/saveliymazovatov/inf111/inference")
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
INFERENCE_EVERY_N_EPOCHS = 1   # поставьте 2-3, если станет тормозить

inference_paths = sorted(p for p in INFERENCE_DIR.iterdir() if p.suffix.lower() in IMG_EXTS)
print(f"Треким предсказания по эпохам на {len(inference_paths)} картинках")


@torch.no_grad()
def log_inference_samples(model, paths, transform, classes, device, logger, epoch, threshold=0.5, max_heatmaps=3):
    model.eval()
    for path in paths:
        image = Image.open(path).convert("RGB")
        img_size = image.size[::-1]  # (H, W) для интерполяции обратно к оригиналу
        tensor = transform(image).unsqueeze(0).to(device)

        logits, attn_map = model(tensor, return_attn=True)   # attn_map: [1, num_classes, h, w]
        probs = torch.sigmoid(logits)[0].cpu().numpy()
        detected = [c for c, p in zip(classes, probs) if p > threshold]

        # какие классы показываем тепловой картой: обнаруженные, либо топ-1 если ничего не сработало
        ranked = sorted(zip(classes, probs), key=lambda x: -x[1])
        heat_classes = detected if detected else [ranked[0][0]]
        heat_classes = heat_classes[:max_heatmaps]

        n_heat = len(heat_classes)
        fig = plt.figure(figsize=(4 * max(2, n_heat), 7))
        gs = GridSpec(2, max(2, n_heat), figure=fig)

        ax_img = fig.add_subplot(gs[0, 0])
        ax_img.imshow(image)
        ax_img.axis("off")
        ax_img.set_title(", ".join(detected) if detected else "Clean", fontsize=10)

        ax_bar = fig.add_subplot(gs[0, 1:])
        colors = ["#d84b30" if p > threshold else "#888780" for p in probs]
        ax_bar.barh(classes, probs, color=colors)
        ax_bar.axvline(threshold, color="gray", linestyle="--", linewidth=1)
        ax_bar.set_xlim(0, 1)
        ax_bar.invert_yaxis()

        img_np = np.array(image)
        for i, cls_name in enumerate(heat_classes):
            idx = classes.index(cls_name)
            heat = attn_map[0, idx].unsqueeze(0).unsqueeze(0)                     # [1,1,h,w]
            heat = F.interpolate(heat, size=img_size, mode="bilinear", align_corners=False)[0, 0]
            heat = heat.cpu().numpy()
            heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)

            ax_heat = fig.add_subplot(gs[1, i])
            ax_heat.imshow(img_np)
            ax_heat.imshow(heat, cmap="jet", alpha=0.45)
            ax_heat.axis("off")
            ax_heat.set_title(f"{cls_name} ({dict(zip(classes, probs))[cls_name]:.2f})", fontsize=9)

        plt.tight_layout()
        logger.report_matplotlib_figure(
            title="Inference",
            series=path.name,
            figure=fig,
            iteration=epoch,
        )
        plt.close(fig)


best_val_loss = float("inf")

epoch_bar = tqdm(range(1, NUM_EPOCHS + 1), desc="Epochs")
for epoch in epoch_bar:
    model.train()
    model.encoder.eval()
    train_loss = 0.0
    grad_norms = []
    batch_bar = tqdm(train_loader, desc=f"  train {epoch}/{NUM_EPOCHS}", leave=False)
    for images, targets, mask in batch_bar:
        images, targets, mask = images.to(device), targets.to(device), mask.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = focal_loss(logits, targets)
        loss.backward()

        grad_norms.append(compute_grad_norm(model))
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()
        train_loss += loss.item() * images.size(0)
        batch_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= len(train_dataset)
    mean_grad_norm = float(np.mean(grad_norms))
    current_lr = optimizer.param_groups[0]["lr"]
    logger.report_scalar("Loss", "train", value=train_loss, iteration=epoch)
    logger.report_scalar("Grad norm", "mean", value=mean_grad_norm, iteration=epoch)
    logger.report_scalar("LR", "value", value=current_lr, iteration=epoch)
    log = f"Epoch {epoch}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | grad_norm={mean_grad_norm:.4f} | lr={current_lr:.2e}"
    postfix = {"train_loss": f"{train_loss:.4f}"}
    val_f1 = None
    val_loss = None
    if test_loader is not None:
        val_loss, macro, per_class = evaluate(model, test_loader, CLASSES, device, focal_loss)
        val_f1 = macro["f1"]

        logger.report_scalar("Loss", "val", value=val_loss, iteration=epoch)
        logger.report_scalar("F1 macro", "val", value=macro["f1"], iteration=epoch)
        logger.report_scalar("Precision macro", "val", value=macro["precision"], iteration=epoch)
        logger.report_scalar("Recall macro", "val", value=macro["recall"], iteration=epoch)
        logger.report_scalar("Accuracy macro", "val", value=macro["accuracy"], iteration=epoch)

        for cls_name, m in per_class.items():
            logger.report_scalar("F1 per-class", cls_name, value=m["f1"], iteration=epoch)
            logger.report_scalar("Precision per-class", cls_name, value=m["precision"], iteration=epoch)
            logger.report_scalar("Recall per-class", cls_name, value=m["recall"], iteration=epoch)
            logger.report_scalar("Accuracy per-class", cls_name, value=m["accuracy"], iteration=epoch)

        log += f" | val_loss={val_loss:.4f} | val_f1_macro={macro['f1']:.4f}"
        postfix.update({"val_loss": f"{val_loss:.4f}", "val_f1": f"{macro['f1']:.4f}"})

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_f1_macro": val_f1,
            }, os.path.join(CHECKPOINT_DIR, "best.pt"))
            log += " | ★ best"

    # --- инференс на контрольных картинках, логируется в ClearML на каждой эпохе ---
    if inference_paths and epoch % INFERENCE_EVERY_N_EPOCHS == 0:
        log_inference_samples(model, inference_paths, test_transform, CLASSES, device, logger, epoch)

    ckpt_path = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch:02d}.pt")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_f1_macro": val_f1,
    }, ckpt_path)
    log += f" | saved: {ckpt_path}"

    epoch_bar.set_postfix(**postfix)
    print(log)

torch.save(model.state_dict(), "/kaggle/working/occlusion_classifier_fpn.pt")
task.close()